# 05 — Visualisations des tables de la database RSNA

Ce notebook regarde `data/database.sqlite`, la base active remplie avec le dataset RSNA Pneumonia Detection Challenge (celui demande par le cahier des charges).

Kaggle (`fkarimovv/abnormal-lung`) a servi de premier dataset pour demarrer le pipeline rapidement (import simple, pas d'authentification particuliere). RSNA est le dataset officiel ; ses resultats sont dans cette base-ci, separee de `data/database_kaggle_dataset.sqlite` (voir notebook 04).

Pre-requis avant d'executer ce notebook :
1. `python scripts/import_rsna_pneumonia.py --max-cases 100` (necessite un compte Kaggle authentifie ayant accepte les regles de la competition `rsna-pneumonia-detection-challenge`)
2. `python scripts/setup_db.py --all` (cree les tables et charge `data/cases.csv` + les prompts)
3. Lancer l'evaluation MedGemma sur ces cas via `scripts/run_prompt_evaluation.py`

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
from src.database import get_runs, get_evaluations, get_cases, get_prompts
import pandas as pd

DB_PATH = Path('..') / 'data' / 'database.sqlite'
BRUTES_RSA = Path('..') / 'data' / 'brutes_rsa'

### Images disponibles

In [ ]:
total = len(list(BRUTES_RSA.glob('*.png'))) if BRUTES_RSA.exists() else 0
print(f"Images RSNA disponibles dans {BRUTES_RSA} : {total}")

### `cases` table

In [ ]:
cases_df = pd.DataFrame(get_cases(DB_PATH))
display(cases_df)
cases_df['ground_truth_label'].value_counts()

### `runs` table

In [ ]:
display(pd.DataFrame(get_runs(DB_PATH)))

### `evaluations` table (jointure runs + evaluations)

In [ ]:
eval_df = pd.DataFrame(get_evaluations(DB_PATH))
display(eval_df)

### `prompts` table

In [ ]:
display(pd.DataFrame(get_prompts(DB_PATH)))

### Metriques par prompt (memes metriques que le notebook 02, voir son glossaire pour le detail)

In [ ]:
from src.metrics import summarize_metrics

prompt_keys = eval_df[['prompt_id', 'prompt_name', 'prompt_version']].drop_duplicates().sort_values('prompt_id')

for _, row in prompt_keys.iterrows():
    df_version = eval_df[eval_df['prompt_id'] == row['prompt_id']]
    rows = [{"label": t, "predicted_class": p, "warning": True}
            for t, p in zip(df_version['ground_truth_label'], df_version['predicted_class'])]
    m = summarize_metrics(rows)
    print(f"\n=== {row['prompt_name']} {row['prompt_version']} ===")
    print(f"Total : {m['n']}")
    print(f"Accuracy : {m['accuracy']:.1%}")
    print(f"Macro F1 : {m['macro_f1']:.3f}")
    print(f"Sensibilité : {m['sensitivity']:.1%}")
    print(f"Spécificité : {m['specificity']:.1%}")